# Irish ABC tune fine-tuning (Qwen3.5)

Fine-tune on your ABC tune CSV, then generate one in a chosen key/meter/length.
Run cells top to bottom. Needs a GPU runtime.

## 1. Install

In [24]:
!pip install -q transformers peft bitsandbytes accelerate datasets


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 2. Load your dataset

Expects a CSV with columns: `abc_notation, control_code, meter, key, note_length`.
Upload it to the Colab file browser (or mount Drive) and set the path below.

In [25]:
from datasets import load_dataset

csv_path = "irishman_clean.csv"  # <-- change to wherever you uploaded the CSV

dataset = load_dataset("csv", data_files=csv_path, split="train")
dataset = dataset.shuffle(seed=42).select(range(int(0.002 * len(dataset))))
print(dataset.column_names)
print(dataset[0])

dataset = dataset.rename_column("abc_notation", "text")

['abc_notation', 'control_code', 'meter', 'key', 'note_length']
{'abc_notation': 'X:114452\nL:1/8\nQ:1/4=120\nM:4/4\nK:G\n"^Allegro"{B} dcBA G2 G2 | G2 BG FGAe |{e} dcBA G2 d2 | efge{e} dcBc | dcBA G2 G2 | G2 BG FGAe | \n dcBA G2 d2 | efge dcBd || g3 a bgfg | afdf afdf | g3 a bgfg | afdf g2{agf} g2 | g3 a bgfg | \n afdf afdf | gabg afdf | gfed efge ||', 'control_code': 'S:2\nB:8\nE:3\nB:8\n', 'meter': '4/4', 'key': 'G', 'note_length': '1/8'}


## 3. Load base model + LoRA

In [26]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)

base_model = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
    device_map="auto",
    trust_remote_code=True,
)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 257.68it/s]


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


## 4. Fine-tune

Adjust `num_train_epochs` based on how many rows you have — start at 1 for a large dataset.

In [27]:
tokenized = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512, padding="max_length"),
    batched=True,
)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="lora-out",
        per_device_train_batch_size=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=50,
        save_strategy="epoch",
        bf16=True,
        report_to="none",
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

model.save_pretrained("lora-out")
tokenizer.save_pretrained("lora-out")

Map: 100%|██████████| 428/428 [00:00<00:00, 2631.68 examples/s]


Step,Training Loss
50,2.097204
100,1.778041
150,1.549086
200,1.507751


('lora-out/tokenizer_config.json',
 'lora-out/chat_template.jinja',
 'lora-out/tokenizer.json')

## 5. Generate a tune

Set the key/meter/note-length/bars you want, then run.

In [31]:
key = "D"
meter = "2/4"
note_length = "2/4"
bars = 16

prompt = f"X:1\nL:{note_length}\nM:{meter}\nK:{key}\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

out = model.generate(**inputs, max_new_tokens=300, do_sample=True, temperature=0.9, top_p=0.95, pad_token_id=tokenizer.eos_token_id)
text = tokenizer.decode(out[0], skip_special_tokens=True)

# Cut off at the Nth bar line so length roughly matches what was asked for.
# Small models can't reliably count bars themselves, so this is done by hand.
parts = text.split("|")
kept = parts[: bars + 1]  # +1 accounts for the header taking the first "cell"
print("|".join(kept))

X:1
L:2/4
M:2/4
K:D
|: D3 B | A3 A | A3 G | G3 F | F2 FG | FEFG | B2 F2 | D2 :|1 
 D2 E2 | D3 A | B3 A | A3 G | G2 FG | FEFG | B2 F2 | D2 :
